In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import urllib.request
import ssl
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, mean_squared_error
from tqdm import tqdm

# ==========================================
# 1. 基础配置 (Configuration)
# ==========================================

class Config:
    """针对 Junyi 数据集优化的实验配置"""
    def __init__(self):
        self.num_students = 0
        self.num_questions = 0
        self.num_concepts = 0

        # 模型参数
        self.embed_dim = 128  # Junyi 数据较为复杂，建议调大维度
        self.seq_len = 100
        self.tcan_layers = 3
        self.kernel_size = 3
        self.dropout = 0.4

        # 训练参数
        self.batch_size = 64
        self.epochs = 20
        self.lr = 0.001
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # 数据集路径 - Junyi 数据集
        self.data_dir = "./data_junyi"
        self.dataset_name = "junyi"
        self.fallback_url = "https://raw.githubusercontent.com/bigdata-ustc/EduData/master/static/junyi/junyi_problem_log.csv"

# ==========================================
# 2. 数据处理模块 (Data Processing for Junyi)
# ==========================================

class JunyiDataset(Dataset):
    def __init__(self, q_ids, labels, s_ids, max_len):
        self.q_ids = q_ids
        self.labels = labels
        self.s_ids = s_ids
        self.max_len = max_len

    def __len__(self):
        return len(self.q_ids)

    def __getitem__(self, idx):
        q_seq = self.q_ids[idx]
        r_seq = self.labels[idx]
        s_id = self.s_ids[idx]
        seq_len = len(q_seq)

        if seq_len >= self.max_len:
            q_seq, r_seq = q_seq[-self.max_len:], r_seq[-self.max_len:]
            mask = np.ones(self.max_len, dtype=np.float32)
        else:
            pad_len = self.max_len - seq_len
            q_seq = np.pad(q_seq, (pad_len, 0), 'constant', constant_values=0)
            r_seq = np.pad(r_seq, (pad_len, 0), 'constant', constant_values=0)
            mask = np.concatenate([np.zeros(pad_len), np.ones(seq_len)]).astype(np.float32)

        return (
            torch.tensor(q_seq, dtype=torch.long),
            torch.tensor(r_seq, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32),
            torch.tensor(s_id, dtype=torch.long)
        )

class DataProcessorJunyi:
    def __init__(self, config):
        self.config = config
        if not os.path.exists(config.data_dir):
            os.makedirs(config.data_dir)

    def download_data(self):
        csv_file = self._find_csv(self.config.data_dir)
        if csv_file: return csv_file

        print("正在安装 EduData 并尝试下载 Junyi 数据集...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "EduData"])
        from EduData import get_data
        try:
            get_data(self.config.dataset_name, self.config.data_dir)
        except Exception:
            save_path = os.path.join(self.config.data_dir, "junyi_problem_log.csv")
            context = ssl._create_unverified_context()
            req = urllib.request.Request(self.config.fallback_url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, context=context) as response, open(save_path, 'wb') as out:
                out.write(response.read())
        return self._find_csv(self.config.data_dir)

    def _find_csv(self, root_dir):
        for root, _, files in os.walk(root_dir):
            for file in files:
                if file.endswith(".csv") and "log" in file.lower():
                    return os.path.join(root, file)
        return None

    def _extract_pseudo_skill(self, name):
        """通过题目名称前缀自动归纳知识点"""
        parts = str(name).split('_')
        return "_".join(parts[:2]) if len(parts) > 1 else parts[0]

    def process(self, order_strategy="chronological"):
        file_path = self.download_data()
        print("正在进行 Junyi 数据集预处理与知识点映射...")

        # 为了运行速度，读取前 80 万行
        df = pd.read_csv(file_path, low_memory=False, nrows=800000)

        # 列名匹配
        cols = df.columns.tolist()
        user_col = next((c for c in cols if c.lower() in ['user_id', 'uid']), 'user_id')
        item_col = next((c for c in cols if c.lower() in ['exercise_name', 'exercise']), 'exercise_name')
        correct_col = next((c for c in cols if c.lower() in ['correct', 'is_correct']), 'correct')
        time_col = next((c for c in cols if c.lower() in ['time_done', 'timestamp']), 'time_done')

        df.dropna(subset=[user_col, item_col, correct_col], inplace=True)
        df[correct_col] = (df[correct_col] > 0).astype(int)

        # 映射题目与提取伪知识点
        df['pseudo_skill'] = df[item_col].apply(self._extract_pseudo_skill)

        user_map = {val: i+1 for i, val in enumerate(df[user_col].unique())}
        prob_map = {val: i+1 for i, val in enumerate(df[item_col].unique())}
        skill_map = {val: i+1 for i, val in enumerate(df['pseudo_skill'].unique())}

        df['user_id_mapped'] = df[user_col].map(user_map)
        df['item_id'] = df[item_col].map(prob_map)
        df['skill_id'] = df['pseudo_skill'].map(skill_map)

        self.config.num_students = len(user_map) + 1
        self.config.num_questions = len(prob_map) + 1
        self.config.num_concepts = len(skill_map) + 1

        # 构建题目-知识点 Q 矩阵
        q_k_relation = np.zeros((self.config.num_questions, self.config.num_concepts))
        tmp_df = df[['item_id', 'skill_id']].drop_duplicates()
        for _, row in tmp_df.iterrows():
            q_k_relation[int(row['item_id']), int(row['skill_id'])] = 1

        # 归一化
        row_sums = q_k_relation.sum(axis=1)
        row_sums[row_sums == 0] = 1
        q_k_relation = q_k_relation / row_sums[:, np.newaxis]

        # 计算难度
        item_diff = df.groupby('item_id')[correct_col].mean()
        diff_values = np.zeros(self.config.num_questions)
        for iid, acc in item_diff.items():
            diff_values[iid] = 1.0 - acc
        df['difficulty_score'] = df['item_id'].map(lambda x: 1.0 - item_diff.get(x, 0.5))

        # 构造训练序列 (取前 1500 名用户)
        all_q, all_r, all_s = [], [], []
        target_users = df['user_id_mapped'].unique()[:1500]
        df_sub = df[df['user_id_mapped'].isin(target_users)]

        for uid, group in tqdm(df_sub.groupby('user_id_mapped'), desc=f"策略: {order_strategy}"):
            if len(group) < 5: continue

            if order_strategy == "chronological":
                group = group.sort_values(by=time_col)
            elif order_strategy == "reverse":
                group = group.sort_values(by=time_col, ascending=False)
            elif order_strategy == "random":
                group = group.sample(frac=1, random_state=42)
            elif order_strategy == "difficulty_asc":
                group = group.sort_values(by='difficulty_score')
            elif order_strategy == "skill_grouped":
                group = group.sort_values(by=['skill_id', time_col])

            all_q.append(group['item_id'].values)
            all_r.append(group[correct_col].values)
            all_s.append(uid)

        return all_q, all_r, all_s, q_k_relation, diff_values

# ==========================================
# 3. 模型定义 (Hetero-Graph + TCAN)
# ==========================================

class HeteroGraphEmbedding(nn.Module):
    def __init__(self, config, diff_values):
        super(HeteroGraphEmbedding, self).__init__()
        self.question_emb = nn.Embedding(config.num_questions, config.embed_dim, padding_idx=0)
        self.concept_emb = nn.Embedding(config.num_concepts, config.embed_dim, padding_idx=0)
        self.register_buffer('diff_values', torch.tensor(diff_values, dtype=torch.float32))
        self.diff_proj = nn.Linear(1, config.embed_dim)

        # 门控融合层
        self.gate = nn.Sequential(nn.Linear(config.embed_dim * 2, config.embed_dim), nn.Sigmoid())
        self.proj = nn.Linear(config.embed_dim * 2, config.embed_dim)

    def forward(self, question_ids, q_k_relation):
        k_emb_weight = self.concept_emb.weight
        q_k_agg = torch.matmul(q_k_relation, k_emb_weight)

        q_base = self.question_emb(question_ids)
        d_feat = self.diff_proj(self.diff_values[question_ids].unsqueeze(-1))

        q_k_feat = F.embedding(question_ids, q_k_agg, padding_idx=0)
        combined = torch.cat([q_base + d_feat, q_k_feat], dim=-1)

        g = self.gate(combined)
        return g * self.proj(combined)

class TemporalAttention(nn.Module):
    def __init__(self, embed_dim, dropout=0.1):
        super(TemporalAttention, self).__init__()
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, L, D = x.size()
        Q, K, V = self.query(x), self.key(x), self.value(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)
        mask = torch.tril(torch.ones(L, L, device=x.device))
        scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(self.dropout(attn), V)

class TCANBlock(nn.Module):
    def __init__(self, embed_dim, kernel_size, dilation, dropout):
        super(TCANBlock, self).__init__()
        self.ta = TemporalAttention(embed_dim, dropout)
        self.conv = nn.Conv1d(embed_dim, embed_dim, kernel_size, dilation=dilation, padding=(kernel_size-1)*dilation)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        residual = x
        z = self.ta(x)
        conv_out = self.conv(z.permute(0, 2, 1))[:, :, :z.size(1)].permute(0, 2, 1)
        return self.dropout(self.relu(self.norm(conv_out + residual)))

class HIG_TCAN_CD_Model(nn.Module):
    def __init__(self, config, q_k_relation, diff_values):
        super(HIG_TCAN_CD_Model, self).__init__()
        self.config = config
        self.register_buffer('q_k_relation', torch.tensor(q_k_relation, dtype=torch.float32))
        self.graph_emb = HeteroGraphEmbedding(config, diff_values)
        self.student_emb = nn.Embedding(config.num_students, config.embed_dim, padding_idx=0)

        self.input_proj = nn.Linear(config.embed_dim + 1, config.embed_dim)
        self.tcan_layers = nn.ModuleList([
            TCANBlock(config.embed_dim, config.kernel_size, 2**i, config.dropout)
            for i in range(config.tcan_layers)
        ])

        self.pred_mlp = nn.Sequential(
            nn.Linear(config.embed_dim * 3, 256), nn.ReLU(), nn.Dropout(config.dropout),
            nn.Linear(256, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, q_ids, answers, student_ids):
        q_emb = self.graph_emb(q_ids, self.q_k_relation)
        x = self.input_proj(torch.cat([q_emb, answers.unsqueeze(-1)], dim=-1))
        h = x
        for layer in self.tcan_layers: h = layer(h)
        s_static = self.student_emb(student_ids).unsqueeze(1).expand(-1, q_ids.size(1), -1)
        return h, q_emb, s_static

# ==========================================
# 4. 训练与消融实验逻辑
# ==========================================

def train_and_eval(config, data_bundle):
    train_loader, test_loader, q_k_rel, diff_values = data_bundle
    model = HIG_TCAN_CD_Model(config, q_k_rel, diff_values).to(config.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)

    best_metrics = {'auc': 0, 'acc': 0, 'rmse': 0}

    for _ in range(config.epochs):
        model.train()
        for q, r, mask, s in train_loader:
            q, r, mask, s = q.to(config.device), r.to(config.device), mask.to(config.device), s.to(config.device)
            optimizer.zero_grad()
            h_seq, q_seq_emb, s_static = model(q, r, s)
            feat = torch.cat([h_seq[:, :-1, :], q_seq_emb[:, 1:, :], s_static[:, 1:, :]], dim=-1)
            preds = model.pred_mlp(feat).squeeze(-1)
            loss = (F.binary_cross_entropy(preds, r[:, 1:], reduction='none') * mask[:, 1:]).sum() / (mask[:, 1:].sum() + 1e-8)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        all_p, all_t = [], []
        with torch.no_grad():
            for q, r, mask, s in test_loader:
                q, r, mask, s = q.to(config.device), r.to(config.device), mask.to(config.device), s.to(config.device)
                h, qe, ss = model(q, r, s)
                feat = torch.cat([h[:, :-1, :], qe[:, 1:, :], ss[:, 1:, :]], dim=-1)
                p = model.pred_mlp(feat).squeeze(-1)
                m = mask[:, 1:].bool()
                all_p.extend(p[m].cpu().numpy())
                all_t.extend(r[:, 1:][m].cpu().numpy())

        auc = roc_auc_score(all_t, all_p)
        if auc > best_metrics['auc']:
            best_metrics = {
                'auc': auc,
                'acc': accuracy_score(all_t, np.array(all_p) > 0.5),
                'rmse': np.sqrt(mean_squared_error(all_t, all_p))
            }

    return best_metrics

def run_junyi_ablation():
    config = Config()
    processor = DataProcessorJunyi(config)
    results = {}

    # 消融策略：原有 5 种策略
    strategies = ["chronological", "reverse", "random", "difficulty_asc", "skill_grouped"]

    for strategy in strategies:
        print(f"\n[运行中] 实验序列顺序策略: {strategy}")
        q_seqs, r_seqs, s_seqs, q_k_rel, diff_values = processor.process(order_strategy=strategy)

        t_q, v_q, t_r, v_r, t_s, v_s = train_test_split(q_seqs, r_seqs, s_seqs, test_size=0.2, random_state=42)

        train_loader = DataLoader(JunyiDataset(t_q, t_r, t_s, config.seq_len), batch_size=config.batch_size, shuffle=True)
        test_loader = DataLoader(JunyiDataset(v_q, v_r, v_s, config.seq_len), batch_size=config.batch_size)

        metrics = train_and_eval(config, (train_loader, test_loader, q_k_rel, diff_values))
        results[strategy] = metrics
        print(f"--- 结果: AUC={metrics['auc']:.4f}, ACC={metrics['acc']:.4f}, RMSE={metrics['rmse']:.4f} ---")

    print("\n" + "="*70)
    print(f"{'TCAN 消融实验结果 (Junyi 数据集)':^70}")
    print("-" * 70)
    print(f"{'策略名称':<25} | {'AUC':<10} | {'ACC':<10} | {'RMSE':<10}")
    print("-" * 70)
    for k in sorted(results, key=lambda x: results[x]['auc'], reverse=True):
        m = results[k]
        print(f"{k:<25} | {m['auc']:.4f}     | {m['acc']:.4f}     | {m['rmse']:.4f}")
    print("="*70)

if __name__ == "__main__":
    run_junyi_ablation()


[运行中] 实验序列顺序策略: chronological
正在安装 EduData 并尝试下载 Junyi 数据集...


downloader, INFO http://base.ustc.edu.cn/data/JunyiAcademy_Math_Practicing_Log/junyi.rar is saved as data_junyi/junyi.rar


downloader, INFO data_junyi/junyi.rar is unrar to data_junyi/junyi



正在进行 Junyi 数据集预处理与知识点映射...


策略: chronological: 100%|██████████| 1500/1500 [00:00<00:00, 2167.70it/s]


...

--- 结果: AUC=0.6920, ACC=0.8390, RMSE=0.3588 ---

[运行中] 实验序列顺序策略: difficulty_asc
正在进行 Junyi 数据集预处理与知识点映射...


策略: difficulty_asc: 100%|██████████| 1500/1500 [00:00<00:00, 2182.50it/s]


--- 结果: AUC=0.6690, ACC=0.8226, RMSE=0.3723 ---

[运行中] 实验序列顺序策略: skill_grouped
正在进行 Junyi 数据集预处理与知识点映射...


策略: skill_grouped: 100%|██████████| 1500/1500 [00:01<00:00, 1151.25it/s]


--- 结果: AUC=0.6860, ACC=0.8328, RMSE=0.3639 ---

                       TCAN 消融实验结果 (Junyi 数据集)                        
----------------------------------------------------------------------
策略名称                      | AUC        | ACC        | RMSE      
----------------------------------------------------------------------
random                    | 0.6920     | 0.8390     | 0.3588
reverse                   | 0.6866     | 0.8382     | 0.3585
skill_grouped             | 0.6860     | 0.8328     | 0.3639
chronological             | 0.6845     | 0.8379     | 0.3603
difficulty_asc            | 0.6690     | 0.8226     | 0.3723
